# OffScript — Phase 1: Data Quality Check
    Ensures data is clean before modeling

In [ ]:
# notebooks/04_data_quality.ipynb

import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pitch_analysis import load_data

data = load_data()
print(f"Total pitches loaded: {len(data)}")
print(f"Pitchers: {data['pitcher_name'].nunique()}")
print(f"Date range: {data['game_date'].min()} to {data['game_date'].max()}")

In [ ]:
# Identify columns with missing values
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print("=== Missing Value Report ===")
print(missing_report)

In [ ]:
# Check for unknown or rare pitch type codes
print("=== Pitch Type Distribution ===")
print(data['pitch_type'].value_counts())

# Flag pitch types that are too rare to model reliably
pitch_counts = data['pitch_type'].value_counts()
rare_pitches = pitch_counts[pitch_counts < 50].index.tolist()
print(f"\nRare pitch types (< 50 occurrences): {rare_pitches}")

In [ ]:
# Pitch locations that are clearly erroneous
print("=== Location Ranges ===")
print(f"plate_x range: {data['plate_x'].min():.2f} to {data['plate_x'].max():.2f}")
print(f"plate_z range: {data['plate_z'].min():.2f} to {data['plate_z'].max():.2f}")

# Flag extreme outliers — pitches logged way outside any realistic location
location_outliers = data[
    (data['plate_x'].abs() > 4) | 
    (data['plate_z'] < 0) | 
    (data['plate_z'] > 6)
]
print(f"\nLocation outliers: {len(location_outliers)} pitches")

In [ ]:
# Ensure each pitcher has enough data to model reliably
pitcher_counts = data.groupby('pitcher_name').size().sort_values()
print("=== Pitches Per Pitcher ===")
print(pitcher_counts)

# Flag anyone with less than 500 pitches
low_volume = pitcher_counts[pitcher_counts < 500]
if len(low_volume) > 0:
    print(f"\nLow volume pitchers (< 500 pitches): {low_volume.index.tolist()}")
else:
    print("\nAll pitchers have sufficient volume for modeling")

In [ ]:
# Apply cleaning decisions based on checks above
data_clean = data.copy()

# Remove rows with missing pitch type — cannot model without this
data_clean = data_clean.dropna(subset=['pitch_type'])

# Remove rare pitch types that cannot be modeled reliably
if rare_pitches:
    data_clean = data_clean[~data_clean['pitch_type'].isin(rare_pitches)]

# Remove location outliers
data_clean = data_clean[
    (data_clean['plate_x'].abs() <= 4) &
    (data_clean['plate_z'] >= 0) &
    (data_clean['plate_z'] <= 6)
]

# Fill missing baserunner columns with 0 — no runner means empty base
for col in ['on_1b', 'on_2b', 'on_3b']:
    data_clean[col] = data_clean[col].fillna(0)

print(f"Pitches before cleaning: {len(data)}")
print(f"Pitches after cleaning: {len(data_clean)}")
print(f"Pitches removed: {len(data) - len(data_clean)}")

# Save cleaned dataset
data_clean.to_parquet('../data/processed/pitcher_data_clean.parquet', index=False)
print("\nClean data saved to data/processed/pitcher_data_clean.parquet")